In [0]:
create schema if not exists 03_gold_catalog.analytics

In [0]:
select count(*) from `03_gold_catalog`.dimensions.dim_customer

In [0]:
with cte1 as(
select account_created_date,row_number() over(order by account_created_date) as rn from `03_gold_catalog`.dimensions.dim_customer
)
select * from cte1 where rn=1 or rn=10000



In [0]:
with cte1 as(
  select customer_sk,min(start_date) as first_start
  from `03_gold_catalog`.facts.fact_opportunity
  group by customer_sk
)
select count(*) as customers_gained
from cte1
where first_start between "2023-01-01" and "2023-03-01"

In [0]:
with ended_customers as(
  select distinct customer_sk
  from `03_gold_catalog`.facts.fact_opportunity
  where end_date between "2023-01-01" and " 2023-03-01"
),
active_customers as(
  select distinct customer_sk
  from `03_gold_catalog`.facts.fact_opportunity
  where start_date > "2023-03-01"
)
select count(*) as lost_customers
from ended_customers e 
left join active_customers a on e.customer_sk = a.customer_sk
where a.customer_sk is null


In [0]:
select customer_sk,count(*) from `03_gold_catalog`.dimensions.dim_customer
group by customer_sk

In [0]:
with repetive_customer as(
  select customer_sk  , count(distinct opportunity_id ) as no_of_times_repeated , sum(revenue_amount) as tot_revenue
  from `03_gold_catalog`.facts.fact_opportunity
  group by customer_sk
)
select d.customer_id,d.customer_name ,r.tot_revenue, r.customer_sk
from `03_gold_catalog`.dimensions.dim_customer as d
join repetive_customer as r on d.customer_sk = r.customer_sk
where no_of_times_repeated > 1
order by tot_revenue desc
limit 10

In [0]:
with customer_revenue as (
  select 
    customer_sk,
    sum(
      revenue_amount / 
      (datediff(end_date, start_date) / 30.0)
    ) as mrr
  from 03_gold_catalog.facts.fact_opportunity
  where end_date > start_date
  group by customer_sk
)

select 
  d.customer_id,
  d.customer_name,
  r.mrr
from 03_gold_catalog.dimensions.dim_customer d
join customer_revenue r
  on d.customer_sk = r.customer_sk
order by r.mrr desc
limit 10;

In [0]:
with cte1 as(
select customer_sk,year(start_date)as year,count(*) as no_of_renewals
from 03_gold_catalog.facts.fact_opportunity
group by customer_sk,year
order by customer_sk,year
),
cte2 as(
  select * , lag(no_of_renewals) over(partition by customer_sk order by year) as prev_year_renewals
  from cte1
),
cte3 as(
  select *,
    case 
      when prev_year_renewals is null then null
      when no_of_renewals > prev_year_renewals then 1
      when no_of_renewals < prev_year_renewals then -1
      else 0
    end as trend
  from cte2
),
cte4 as (
  select customer_sk ,min(trend) as min_trend ,max(trend) as max_trend
  from cte3
  where trend is not null
  group by customer_sk
),
cte5 as(
select d.customer_id,
  case 
    when min_trend = 1 and max_trend = 1 then 'up'
    when min_trend = -1 and max_trend = -1 then 'down'
    else 'stable'
  end as growth_trend
from cte4 c
join 03_gold_catalog.dimensions.dim_customer d on c.customer_sk = d.customer_sk
order by customer_id
)
select * from cte5 where growth_trend="up"